# ConvNeXt-Tiny Multi-Label Classification Fine-Tuning

This notebook fine-tunes ConvNeXt-Tiny for multi-label fashion attribute classification using:
- **Model**: ConvNeXt-Tiny pre-trained on ImageNet
- **Classification**: Multi-label classification head
- **Loss**: BCEWithLogitsLoss (Binary Cross Entropy)
- **Strategy**: Partial fine-tuning (freeze early layers)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.models import convnext_tiny, ConvNeXt_Tiny_Weights
import numpy as np
import pandas as pd
from sklearn.preprocessing import MultiLabelBinarizer
import json
import os
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from sklearn.metrics import hamming_loss, accuracy_score, f1_score
import warnings
warnings.filterwarnings('ignore')

# Check device availability
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

## Configuration & Hyperparameters

In [ ]:
# Configuration
CONFIG = {
    'batch_size': 32,
    'num_epochs': 20,
    'learning_rate': 1e-4,
    'weight_decay': 1e-5,
    'warmup_epochs': 2,
    'num_workers': 4,
    'image_size': 224,
    'seed': 42,
    'threshold': 0.5,  # For multi-label prediction threshold
    'freeze_until_layer': 4,  # Freeze first 4 stages of ConvNeXt
}

# Set seed for reproducibility
np.random.seed(CONFIG['seed'])
torch.manual_seed(CONFIG['seed'])
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(CONFIG['seed'])

# Data paths
DATA_DIR = Path('/Users/rafiabhista/Documents/code/apple/c1_nlp/fashionpedia_5/fashionpedia_processed')
TRAIN_CSV = DATA_DIR / 'train.csv'
VAL_CSV = DATA_DIR / 'val.csv'
TRAIN_DIR = Path('/Users/rafiabhista/Documents/code/apple/c1_nlp/fashionpedia_5/train')
VAL_DIR = Path('/Users/rafiabhista/Documents/code/apple/c1_nlp/fashionpedia_5/val')

print(f"Configuration: {CONFIG}")

## Custom Dataset Class

In [ ]:
from PIL import Image
import json

class MultiLabelDataset(Dataset):
    """Custom dataset for multi-label image classification"""
    
    def __init__(self, csv_path, image_dir, transform=None, label_type='attributes'):
        """
        Args:
            csv_path: Path to CSV with columns [image_id, attributes/categories]
            image_dir: Directory containing images
            transform: Data augmentation transforms
            label_type: Type of labels to use ('attributes' or 'categories')
        """
        self.df = pd.read_csv(csv_path)
        self.image_dir = Path(image_dir)
        self.transform = transform
        self.label_type = label_type
        
        # Initialize MultiLabelBinarizer
        self.mlb = MultiLabelBinarizer()
        
        # Process labels
        self.df['label_list'] = self.df[label_type].apply(
            lambda x: json.loads(x) if isinstance(x, str) else x
        )
        
        # Fit the binarizer on all labels
        self.mlb.fit(self.df['label_list'])
        self.num_classes = len(self.mlb.classes_)
        
        print(f"Dataset loaded: {len(self.df)} samples, {self.num_classes} classes")
        print(f"Classes: {self.mlb.classes_[:10]}...")  # Show first 10
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        
        # Load image
        img_path = self.image_dir / f"{row['image_id']}.jpg"
        if not img_path.exists():
            img_path = self.image_dir / f"{row['image_id']}.png"
        
        image = Image.open(img_path).convert('RGB')
        
        # Apply transforms
        if self.transform:
            image = self.transform(image)
        
        # Get multi-label target
        labels = row['label_list']
        target = self.mlb.transform([labels])[0].astype(np.float32)
        
        return {
            'image': torch.from_numpy(np.array(image)) if not isinstance(image, torch.Tensor) else image,
            'target': torch.from_numpy(target),
            'image_id': row['image_id']
        }

# Define transforms
train_transform = transforms.Compose([
    transforms.Resize((CONFIG['image_size'], CONFIG['image_size'])),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomAffine(degrees=15, translate=(0.1, 0.1), scale=(0.9, 1.1)),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

val_transform = transforms.Compose([
    transforms.Resize((CONFIG['image_size'], CONFIG['image_size'])),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

## Model Architecture: ConvNeXt-Tiny + Multi-Label Head

In [ ]:
class ConvNeXtMultiLabel(nn.Module):
    """ConvNeXt-Tiny with custom multi-label classification head"""
    
    def __init__(self, num_classes, freeze_until_stage=4):
        """
        Args:
            num_classes: Number of output classes (attributes)
            freeze_until_stage: Stages to freeze (0-4 in ConvNeXt)
        """
        super(ConvNeXtMultiLabel, self).__init__()
        
        # Load pre-trained ConvNeXt-Tiny
        self.backbone = convnext_tiny(weights=ConvNeXt_Tiny_Weights.IMAGENET1K_V1)
        
        # Get the feature dimension from backbone
        feature_dim = self.backbone.classifier[2].in_features
        
        # Freeze early layers (partial fine-tuning)
        self._freeze_backbone_stages(freeze_until_stage)
        
        # Replace classification head with multi-label head
        self.classifier = nn.Sequential(
            nn.LayerNorm(feature_dim),
            nn.Dropout(p=0.5),
            nn.Linear(feature_dim, 512),
            nn.GELU(),
            nn.Dropout(p=0.3),
            nn.Linear(512, 256),
            nn.GELU(),
            nn.Dropout(p=0.3),
            nn.Linear(256, num_classes)
        )
    
    def _freeze_backbone_stages(self, freeze_until_stage):
        """
        Freeze ConvNeXt stages for partial fine-tuning.
        ConvNeXt-Tiny has 4 stages (0-3) + classifier
        """
        # Freeze features up to specified stage
        if freeze_until_stage > 0:
            for i in range(freeze_until_stage):
                for param in self.backbone.features[i].parameters():
                    param.requires_grad = False
                print(f"Frozen stage {i}")
        
        # Freeze normalization layer
        for param in self.backbone.classifier[0].parameters():
            param.requires_grad = False
    
    def forward(self, x):
        # Backbone feature extraction
        features = self.backbone.features(x)
        
        # Global average pooling
        x = self.backbone.avgpool(features)
        x = x.flatten(1)
        
        # Multi-label classification head
        logits = self.classifier(x)
        
        return logits
    
    def get_trainable_params(self):
        """Get number of trainable parameters"""
        return sum(p.numel() for p in self.parameters() if p.requires_grad)
    
    def get_total_params(self):
        """Get total number of parameters"""
        return sum(p.numel() for p in self.parameters())

# Create model
model = ConvNeXtMultiLabel(num_classes=50, freeze_until_stage=CONFIG['freeze_until_layer'])
model = model.to(device)

total_params = model.get_total_params()
trainable_params = model.get_trainable_params()

print(f"\n{'='*60}")
print(f"Model: ConvNeXt-Tiny with Multi-Label Classification Head")
print(f"{'='*60}")
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Frozen parameters: {total_params - trainable_params:,}")
print(f"Trainable: {100 * trainable_params / total_params:.2f}%")
print(f"{'='*60}\n")

## Data Loading